# NCERT RAG Q&A System
Step-by-step walkthrough of a BM25 + Gemini retrieval-augmented generation pipeline for CBSE Class 9 Science.

In [ ]:
import sys
!{sys.executable} -m pip install pdfplumber rank_bm25 transformers nltk requests -q

## 1. Imports & Configuration

In [ ]:
import os, re, pdfplumber, nltk, requests
from nltk.corpus import stopwords
from collections import Counter
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer

nltk.download('stopwords', quiet=True)

PDF_FILE_PATH  = 'ncert-9.pdf'
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')
print('API key set:', bool(GEMINI_API_KEY))

## 2. PDF Text Extraction
Skip the first 12 pages (cover, TOC) and extract chapter content.

In [ ]:
def extract_text_from_pdf(pdf_path):
    try:
        text = ''
        with pdfplumber.open(pdf_path) as doc:
            for page in doc.pages[12:]:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + '\n'
        return text
    except Exception as e:
        print(f'Error: {e}')
        return ''

extracted_text = extract_text_from_pdf(PDF_FILE_PATH)
print(f'Extracted {len(extracted_text):,} characters')
print(extracted_text[:400])

## 3. Keyword Frequency Analysis

In [ ]:
stop_words = set(stopwords.words('english'))
words = re.findall(r'\b\w+\b', extracted_text.lower())
filtered_words = [w for w in words if w not in stop_words]
counter = Counter(filtered_words)
print('Top 20 words:')
for word, count in counter.most_common(20):
    print(f'  {word:<20} {count}')

## 4. Block Splitting & Classification
Split on structural boundaries (examples, exercises, numbered items, headings).

In [ ]:
def is_boundary(line):
    line = line.strip()
    if not line: return False
    lower = line.lower()
    return (
        lower.startswith(('example','illustration','exercise','questions','q.','q '))
        or line.startswith(tuple(str(i)+'.' for i in range(1,20)))
        or (len(line) < 60 and line.isupper())
    )

def split_into_blocks(text):
    blocks, cur = [], ''
    for line in text.split('\n'):
        line = line.strip()
        if not line: continue
        if is_boundary(line):
            if cur: blocks.append(cur.strip()); cur = ''
            blocks.append(line)
        else:
            cur = (cur + ' ' + line).strip() if cur else line
    if cur: blocks.append(cur.strip())
    return blocks

def classify_block(text):
    t = text.lower()
    if text.strip().startswith(tuple(str(i)+'.' for i in range(1,20))): return 'question'
    if any(k in t for k in ['question','questions']): return 'question'
    if any(k in t for k in ['example','illustration','case','sample']): return 'example'
    if '=' in text: return 'formula'
    return 'concept'

def build_structured_blocks(text):
    return [{'text': b, 'type': classify_block(b)} for b in split_into_blocks(text)]

structured_blocks = build_structured_blocks(extracted_text)
type_counts = Counter(b['type'] for b in structured_blocks)
print(f'Total blocks: {len(structured_blocks)}')
print('Type distribution:', dict(type_counts))

for b in structured_blocks[10:14]:
    print(f"\n[{b['type']}] {b['text'][:100]}")

## 5. Token-based Chunking
Sliding window (size=25 tokens, overlap=10) using T5 tokenizer.

In [ ]:
def chunk_blocks(structured_blocks, chunk_size=25, overlap=10):
    tokenizer = AutoTokenizer.from_pretrained('t5-small')
    chunks = []
    for block in structured_blocks:
        text, btype = block['text'], block['type']
        tokens = tokenizer.encode(text)
        if len(tokens) <= chunk_size:
            chunks.append({'text': text, 'type': btype}); continue
        start = 0
        while start < len(tokens):
            ct = tokenizer.decode(tokens[start:start+chunk_size])
            chunks.append({'text': ct, 'type': btype})
            start += chunk_size - overlap
    return chunks

tokenized_chunks = chunk_blocks(structured_blocks)
avg_len = sum(len(c['text'].split()) for c in tokenized_chunks) / len(tokenized_chunks)
print(f'Total chunks: {len(tokenized_chunks)} | Avg length: {avg_len:.1f} words')

for c in tokenized_chunks[:3]:
    print(f"\n[{c['type']}] {c['text']}")

## 6. BM25 Retrieval Index

In [ ]:
def build_bm25_index(chunks):
    return BM25Okapi([c['text'].lower().split() for c in chunks])

def retrieve(query, bm25, chunks, top_k=5):
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [{'text': chunks[i]['text'], 'type': chunks[i]['type'], 'score': scores[i]} for i in ranked[:top_k]]

def is_low_confidence(retrieved_chunks, threshold=0.5):
    return sum(1 for c in retrieved_chunks[:3] if c['score'] < threshold) >= 3

bm25 = build_bm25_index(tokenized_chunks)

test_results = retrieve('What is force?', bm25, tokenized_chunks)
print(f'Low confidence: {is_low_confidence(test_results)}')
for r in test_results:
    print(f"  score={r['score']:.3f}  [{r['type']}] {r['text'][:60]}")

## 7. Prompt Building & Gemini Generation

In [ ]:
def build_prompt(query, chunks, low_confidence=False):
    ctx = ''.join(f"[Chunk {i+1} | {c['type']}]\n{c['text']}\n\n" for i,c in enumerate(chunks))
    instr = ('Context may NOT be reliable. Say I don\'t know if unsure.'
             if low_confidence else
             'Answer ONLY from context. Say I don\'t know if not present.')
    return f'You are a science tutor.\n\n{instr}\n\nContext:\n{ctx}\nQuestion:\n{query}\n\nAnswer:\n'

def generate_answer(prompt, api_key):
    url = ('https://generativelanguage.googleapis.com/v1beta/models/'
           f'gemini-1.5-flash:generateContent?key={api_key}')
    r = requests.post(url, json={'contents': [{'parts': [{'text': prompt}]}],
                                  'generationConfig': {'temperature': 0}}, timeout=30)
    r.raise_for_status()
    return r.json()['candidates'][0]['content']['parts'][0]['text']

def answer(question, bm25, chunks, api_key):
    ret = retrieve(question, bm25, chunks)
    lc  = is_low_confidence(ret)
    return {'question': question, 'answer': generate_answer(build_prompt(question, ret, lc), api_key),
            'chunks': ret, 'low_confidence': lc}

if GEMINI_API_KEY:
    r = answer("State Newton's second law of motion.", bm25, tokenized_chunks, GEMINI_API_KEY)
    print('LOW CONF:', r['low_confidence'])
    print('ANSWER:', r['answer'].strip())
else:
    print('Set GEMINI_API_KEY first.')

## 8. Evaluate — 10 CBSE Questions

In [ ]:
CBSE_QUESTIONS = [
    'What are the characteristics of particles of matter?',
    'What is the difference between a mixture and a compound?',
    'State the law of conservation of mass.',
    'What is the difference between speed and velocity?',
    "State Newton's second law of motion.",
    'What is the universal law of gravitation?',
    'What is kinetic energy? Give its formula.',
    'What is the difference between loudness and intensity of sound?',
    'What are the differences between infectious and non-infectious diseases?',
    'What is the role of the atmosphere in climate control?',
]

if GEMINI_API_KEY:
    for idx, q in enumerate(CBSE_QUESTIONS, 1):
        print(f'\n{"─"*65}')
        print(f'Q{idx}: {q}')
        print('─'*65)
        res = answer(q, bm25, tokenized_chunks, GEMINI_API_KEY)
        if res['low_confidence']:
            print('[⚠] Low confidence')
        print(res['answer'].strip())
else:
    print('Set GEMINI_API_KEY first.')